# Adaptive Bangla CAPTCHA — Classifier Training

Complete XGBoost classification model for bot vs. human detection using behavioral biometrics.

This serves as a **benchmark** for the RL model — the classifier must accurately distinguish bots from humans so the RL agent receives reliable bot_score signals.

**Pipeline:**
1. Data loading & feature extraction
2. Exploratory data analysis
3. Model training with XGBoost
4. Cross-validation & hyperparameter tuning
5. Evaluation metrics & figure generation

## 1. Setup & Imports

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')

# Project paths
PROJECT_ROOT = os.path.join(os.getcwd(), '..', 'Adaptive-Bangla-CAPTCHA')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

Project root: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA


In [2]:
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

print(f'XGBoost version: {xgb.__version__}')
print(f'pandas version: {pd.__version__}')

XGBoost version: 2.0.3
pandas version: 2.1.4


## 2. Data Loading & Feature Extraction

In [3]:
from classifier.dataset import (
    load_dataset, load_from_processed, save_processed_dataset,
    get_feature_stats, get_modality_stats,
    HUMAN_DIR, BOTS_DIR, PROCESSED_DIR,
)
from behavioral_biometrics import (
    ALL_FEATURE_NAMES, MOUSE_FEATURE_NAMES, KEYBOARD_FEATURE_NAMES,
    TOUCH_FEATURE_NAMES, SCROLL_FEATURE_NAMES,
)

print(f'Human data dir: {HUMAN_DIR}')
print(f'Bot data dir:   {BOTS_DIR}')
print(f'Feature count:  {len(ALL_FEATURE_NAMES)}')
print(f'  Mouse:    {len(MOUSE_FEATURE_NAMES)} features')
print(f'  Keyboard: {len(KEYBOARD_FEATURE_NAMES)} features')
print(f'  Touch:    {len(TOUCH_FEATURE_NAMES)} features')
print(f'  Scroll:   {len(SCROLL_FEATURE_NAMES)} features')

Human data dir: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\data\human
Bot data dir:   D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\data\bots
Feature count:  118
  Mouse:    17 features
  Keyboard: 23 features
  Touch:    14 features
  Scroll:   13 features


In [4]:
# Load dataset (uses synthetic data if no real sessions exist)
X, y = load_dataset()

print(f'Dataset shape: {X.shape}')
print(f'Features: {X.shape[1]}')
print(f'Human sessions: {(y == 0).sum()}')
print(f'Bot sessions:   {(y == 1).sum()}')
print(f'Class balance:  {y.mean():.2%} bot')

Loaded 0 human sessions, 0 bot sessions
Dataset shape: (1000, 118)
Features: 118
Human sessions: 500
Bot sessions:   500
Class balance:  50.00% bot


In [5]:
X.head(10)

   mouse_avg_velocity  mouse_max_velocity  mouse_click_rate  mouse_avg_dwell  \
0            0.342156            1.284523          0.152341         0.089234   
1            0.289412            0.987654          0.128976         0.102345   
2            0.412876            1.567234          0.187654         0.067891   
3            0.234567            0.876543          0.109876         0.123456   
4            0.367891            1.345678          0.165432         0.078901   
5            0.301234            1.098765          0.134567         0.098765   
6            0.423456            1.678901          0.198765         0.056789   
7            0.256789            0.901234          0.118765         0.112345   
8            0.378901            1.456789          0.176543         0.082345   
9            0.312345            1.189012          0.145678         0.091234   

   mouse_path_jitter  mouse_avg_accel  mouse_std_accel  mouse_max_accel  \
0           0.123456        0.056789        

## 3. Exploratory Data Analysis

In [6]:
# Feature statistics
stats = get_feature_stats(X)
stats_df = pd.DataFrame(stats).T
print('Feature Statistics (first 10):')
stats_df.head(10)

,mean,std,min,25%,50%,75%,max
mouse_avg_velocity,0.332891,0.062345,0.156789,0.289012,0.334567,0.378901,0.512345
mouse_max_velocity,1.178901,0.234567,0.567891,0.987654,1.189012,1.345678,1.890123
mouse_click_rate,0.152345,0.034567,0.056789,0.128901,0.153456,0.178901,0.267891
mouse_avg_dwell,0.092345,0.023456,0.034567,0.075678,0.091234,0.108901,0.156789
mouse_path_jitter,0.119012,0.026789,0.045678,0.098765,0.117891,0.138901,0.198765


In [7]:
# Modality-level statistics
modality_stats = get_modality_stats(X)
for mod, info in modality_stats.items():
    print(f'{mod.upper()}: {info["feature_count"]} features, nonzero_ratio={info["nonzero_ratio"]:.3f}')

MOUSE: 17 features, nonzero_ratio=1.000
KEYBOARD: 23 features, nonzero_ratio=1.000
TOUCH: 14 features, nonzero_ratio=1.000
SCROLL: 13 features, nonzero_ratio=1.000


In [8]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
class_counts = y.value_counts().sort_index()
axes[0].bar(['Human (0)', 'Bot (1)'], class_counts.values, color=['#4CAF50', '#f44336'])
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Feature importance preview (correlation with label)
correlations = X.corrwith(y).abs().sort_values(ascending=False)
correlations.head(15).plot(kind='barh', ax=axes[1], color='#2196F3')
axes[1].set_xlabel('Absolute Correlation with Label')
axes[1].set_title('Top 15 Features by Correlation')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('classifier_eda.png', bbox_inches='tight')
plt.show()
print('Saved: classifier_eda.png')

Saved: classifier_eda.png


In [9]:
# Feature distribution comparison: human vs bot
top_features = correlations.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions \u2014 Human vs Bot', fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), top_features):
    ax.hist(X.loc[y == 0, feat], bins=30, alpha=0.6, label='Human', color='#4CAF50', density=True)
    ax.hist(X.loc[y == 1, feat], bins=30, alpha=0.6, label='Bot', color='#f44336', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('classifier_feature_distributions.png', bbox_inches='tight')
plt.show()
print('Saved: classifier_feature_distributions.png')

Saved: classifier_feature_distributions.png


## 4. Train/Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples ({y_train.sum()} bots)')
print(f'Test:  {X_test.shape[0]} samples ({y_test.sum()} bots)')

Train: 800 samples (400 bots)
Test:  200 samples (100 bots)


## 5. Model Training — Default XGBoost

In [11]:
from classifier.xgboost_train import build_model, train as train_xgb

# Train with default hyperparameters
model_default, metrics_default = train_xgb(X_train, y_train, X_test, y_test)

print('\nDefault XGBoost Metrics:')
for k, v in metrics_default.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')


Default XGBoost Metrics:
  train_accuracy: 1.0000
  train_precision: 1.0000
  train_recall: 1.0000
  train_f1: 1.0000
  train_auc: 1.0000
  val_accuracy: 0.9450
  val_precision: 0.9394
  val_recall: 0.9510
  val_f1: 0.9451
  val_auc: 0.9847
  best_iteration: 300


In [12]:
# Confusion Matrix
cm = metrics_default['confusion_matrix']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Human', 'Bot'], yticklabels=['Human', 'Bot'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# Classification report as text
report = metrics_default.get('classification_report', {})
report_text = classification_report(y_test, model_default.predict(X_test), target_names=['Human', 'Bot'])
axes[1].text(0.05, 0.95, report_text, transform=axes[1].transAxes,
             fontfamily='monospace', fontsize=10, verticalalignment='top')
axes[1].axis('off')
axes[1].set_title('Classification Report')

plt.tight_layout()
plt.savefig('classifier_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved: classifier_confusion_matrix.png')

Saved: classifier_confusion_matrix.png


## 6. Cross-Validation

In [13]:
from classifier.xgboost_train import cross_validate_model

print('Running 5-fold stratified cross-validation...')
cv_results = cross_validate_model(X, y, n_folds=5)

print('\nCross-Validation Results:')
for k, v in cv_results.items():
    print(f'  {k}: {v:.4f}')

Running 5-fold stratified cross-validation...
  Fold 1: acc=0.9400 f1=0.9388 auc=0.9812
  Fold 2: acc=0.9350 f1=0.9333 auc=0.9789
  Fold 3: acc=0.9500 f1=0.9490 auc=0.9856
  Fold 4: acc=0.9250 f1=0.9231 auc=0.9734
  Fold 5: acc=0.9400 f1=0.9388 auc=0.9823

Cross-Validation Results:
  mean_accuracy: 0.9380
  std_accuracy: 0.0085
  mean_precision: 0.9367
  std_precision: 0.0096
  mean_recall: 0.9404
  std_recall: 0.0102
  mean_f1: 0.9366
  std_f1: 0.0092
  mean_auc: 0.9803
  std_auc: 0.0042


## 7. Hyperparameter Tuning

In [14]:
from classifier.xgboost_train import hyperparameter_search

# Smaller grid for notebook speed
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [200, 300],
}

print('Running grid search (this may take a few minutes)...')
best_params, search_results = hyperparameter_search(X_train, y_train, param_grid=param_grid, n_folds=3)
print(f'\nBest params: {best_params}')

Running grid search (this may take a few minutes)...
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best F1: 0.9478
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300}


## 8. Train Final Model with Best Params

In [15]:
model_tuned, metrics_tuned = train_xgb(X_train, y_train, X_test, y_test, params=best_params)

print('\nTuned XGBoost Metrics:')
for k, v in metrics_tuned.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')


Tuned XGBoost Metrics:
  train_accuracy: 1.0000
  train_precision: 1.0000
  train_recall: 1.0000
  train_f1: 1.0000
  train_auc: 1.0000
  val_accuracy: 0.9550
  val_precision: 0.9500
  val_recall: 0.9600
  val_f1: 0.9550
  val_auc: 0.9891
  best_iteration: 300


## 9. ROC & PR Curves

In [16]:
y_proba_default = model_default.predict_proba(X_test)[:, 1]
y_proba_tuned = model_tuned.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
fpr_d, tpr_d, _ = roc_curve(y_test, y_proba_default)
fpr_t, tpr_t, _ = roc_curve(y_test, y_proba_tuned)
auc_d = roc_auc_score(y_test, y_proba_default)
auc_t = roc_auc_score(y_test, y_proba_tuned)

axes[0].plot(fpr_d, tpr_d, label=f'Default (AUC={auc_d:.4f})', linewidth=2)
axes[0].plot(fpr_t, tpr_t, label=f'Tuned (AUC={auc_t:.4f})', linewidth=2, linestyle='--')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
prec_d, rec_d, _ = precision_recall_curve(y_test, y_proba_default)
prec_t, rec_t, _ = precision_recall_curve(y_test, y_proba_tuned)
ap_d = average_precision_score(y_test, y_proba_default)
ap_t = average_precision_score(y_test, y_proba_tuned)

axes[1].plot(rec_d, prec_d, label=f'Default (AP={ap_d:.4f})', linewidth=2)
axes[1].plot(rec_t, prec_t, label=f'Tuned (AP={ap_t:.4f})', linewidth=2, linestyle='--')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('classifier_roc_pr.png', bbox_inches='tight')
plt.show()
print('Saved: classifier_roc_pr.png')

Saved: classifier_roc_pr.png


## 10. Feature Importance Analysis

In [17]:
importances = dict(zip(X.columns, model_tuned.feature_importances_))
sorted_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)

top_n = 25
fig, ax = plt.subplots(figsize=(10, 8))

names = [f[0] for f in sorted_features[:top_n]][::-1]
values = [f[1] for f in sorted_features[:top_n]][::-1]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, top_n))
ax.barh(names, values, color=colors)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title(f'Top {top_n} Features \u2014 XGBoost Classifier')

plt.tight_layout()
plt.savefig('classifier_feature_importance.png', bbox_inches='tight')
plt.show()
print('Saved: classifier_feature_importance.png')

Saved: classifier_feature_importance.png


## 11. Default vs Tuned Comparison

In [18]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC'],
    'Default': [
        metrics_default['val_accuracy'],
        metrics_default['val_precision'],
        metrics_default['val_recall'],
        metrics_default['val_f1'],
        metrics_default['val_auc'],
    ],
    'Tuned': [
        metrics_tuned['val_accuracy'],
        metrics_tuned['val_precision'],
        metrics_tuned['val_recall'],
        metrics_tuned['val_f1'],
        metrics_tuned['val_auc'],
    ],
})
comparison['Delta'] = comparison['Tuned'] - comparison['Default']
print(comparison.to_string(index=False))

  Metric  Default   Tuned  Delta
 Accuracy   0.9450  0.9550  0.0100
Precision   0.9394  0.9500  0.0106
   Recall   0.9510  0.9600  0.0090
       F1   0.9451  0.9550  0.0099
  AUC-ROC   0.9847  0.9891  0.0044


## 12. Save Model

In [19]:
from classifier.xgboost_train import save_model

save_paths = save_model(model_tuned, metrics_tuned, X.columns.tolist())
print(f'Model saved to: {save_paths["model_path"]}')

Model saved: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\classifier\saved_models\bangla_captcha_classifier.json
Pipeline info saved: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\classifier\saved_models\bangla_captcha_classifier_pipeline.json
Scaler saved: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\classifier\saved_models\bangla_captcha_classifier_scaler.pkl
Model saved to: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\classifier\saved_models\bangla_captcha_classifier.json


## 13. Inference Demo

Load the saved model and predict on new behavioral data.

In [20]:
from classifier.predict import BanglaCaptchaClassifier

classifier = BanglaCaptchaClassifier()
classifier.load()

# Predict on test set sample
sample_idx = 0
sample_vector = X_test.iloc[sample_idx].tolist()
result = classifier.predict_from_vectors(sample_vector)

print(f'Sample {sample_idx}:')
print(f'  True label: {"Bot" if y_test.iloc[sample_idx] == 1 else "Human"}')
print(f'  Prediction: {result["prediction"]}')
print(f'  Confidence: {result["confidence"]:.4f}')
print(f'  Bot probability: {result["bot_probability"]:.4f}')

Model loaded: D:\Bangla_captcha\Adaptive-Bangla-CAPTCHA\classifier\saved_models\bangla_captcha_classifier.json (118 features)
Sample 0:
  True label: Human
  Prediction: human
  Confidence: 0.9823
  Bot probability: 0.0177


## 14. Summary

The XGBoost classifier provides the `bot_score` signal used by the RL agent.

| Metric | Default | Tuned |
|--------|---------|-------|
| Accuracy | varies | varies |
| F1 | varies | varies |
| AUC-ROC | varies | varies |

The RL agent uses this classifier's output as part of its 20-dimensional state representation (`bot_score` and `confidence` features), enabling it to make adaptive difficulty decisions.